In [12]:

import tensorflow as tf
import sympy as sp
from sympy.utilities.lambdify import lambdify
import time

# --- Method 1: Direct TensorFlow Function ---
@tf.function
def direct_function(x):
    # Directly defined TensorFlow expression
    return tf.sin(x) / tf.pow(tf.cos(x), 2)

# --- Method 2: Dynamic Expression with compile() and eval() ---
expression_str = "tf.sin(x) / tf.pow(tf.cos(x), 2)"
# Pre-compile the expression to avoid repeated parsing overhead.
compiled_expr = compile(expression_str, '<string>', 'eval')

@tf.function
def dynamic_function(x):
    # Evaluate the pre-compiled expression.
    return eval(compiled_expr)

# --- Method 3: Lambdify via Sympy ---
# Define the symbolic variable and expression.
x_sym = sp.symbols('x')
expr_sym = sp.sin(x_sym)/ sp.cos(x_sym)**2
# Create a TensorFlow-compatible function using lambdify.
tf_func = lambdify(x_sym, expr_sym, modules=['tensorflow'])

@tf.function
def lambdify_function(x):
    return tf_func(x)

# --- Input tensor ---
x_val = tf.constant(0.5)

# Warm-up calls (to trace and build the static graphs)
print("Direct function output:  ", direct_function(x_val).numpy())
print("Dynamic function output: ", dynamic_function(x_val).numpy())
print("Lambdify function output:", lambdify_function(x_val).numpy())

# --- Benchmarking ---
n_runs = 10000

# Benchmark Direct Function
start = time.time()
for _ in range(n_runs):
    _ = direct_function(x_val)
end = time.time()
direct_avg_time = (end - start) / n_runs

# Benchmark Dynamic Function
start = time.time()
for _ in range(n_runs):
    _ = dynamic_function(x_val)
end = time.time()
dynamic_avg_time = (end - start) / n_runs

# Benchmark Lambdify Function
start = time.time()
for _ in range(n_runs):
    _ = lambdify_function(x_val)
end = time.time()
lambdify_avg_time = (end - start) / n_runs

print("\nAverage execution times per run:")
print("Direct function:  {:.6f} sec".format(direct_avg_time))
print("Dynamic function: {:.6f} sec".format(dynamic_avg_time))
print("Lambdify function:{:.6f} sec".format(lambdify_avg_time))

Direct function output:   0.6225084
Dynamic function output:  0.6225084
Lambdify function output: 0.6225084

Average execution times per run:
Direct function:  0.000508 sec
Dynamic function: 0.000494 sec
Lambdify function:0.000484 sec
